<a href="https://colab.research.google.com/github/jimmyjcarranza/ribot-in-silico/blob/main/notebooks/01_hello_pythia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Hello, Pythia

**Goal of this notebook:** prove that the whole circuit works before building the experiment on top of it.

By the end you will have:

1. Loaded Pythia-160M, the model the whole project uses.
2. Asked it one fact and read its answer as probabilities.
3. Written `score_fact`, the rule that decides whether the model "knows" a fact. Every later stage reuses it.
4. Loaded earlier training snapshots of the same model and watched a fact appear over training.

No experiment results come out of this notebook. It is the smoke test.

**Before running:** in Colab, go to `Runtime > Change runtime type` and pick `T4 GPU`. It also runs on CPU, only slower.

## 1. Setup

**What it does:** imports the libraries and checks whether a GPU is available.

**What you should see:** `device: cuda` on a Colab GPU runtime, or `device: cpu` otherwise.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


## 2. Load the model

**What it does:** downloads Pythia-160M from Hugging Face. EleutherAI published 154 snapshots of this model taken during training. Each one is selected with `revision="stepN"`. Step 143000 is the final, fully trained model.

`load_checkpoint` is the one function that picks a snapshot, so every later notebook goes through it.

**What you should see:** a download bar the first time (about 375 MB), then `layers: 12` and roughly 162 million parameters.

In [2]:
MODEL_NAME = "EleutherAI/pythia-160m"
FINAL_STEP = 143000

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def load_checkpoint(step):
    """Load the Pythia-160M snapshot saved at a given training step."""
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=f"step{step}")
    # The weights are stored in float16. Cast to float32 so the numbers
    # are the same on CPU, on GPU and across library versions.
    return model.float().to(device).eval()


model = load_checkpoint(FINAL_STEP)
print("layers:", model.config.num_hidden_layers)
print("parameters:", sum(p.numel() for p in model.parameters()))

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  649MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

layers: 12
parameters: 162322944


## 3. Ask one fact

**What it does:** a language model does not answer questions. Given some text, it outputs one probability per vocabulary token for what comes next. `next_token_probs` returns that full list and `show_top` prints the five most likely next tokens.

**What you should see:** `' Shakespeare'` in first place with a probability a bit above 0.5. Note the leading space: for the tokenizer, `' Shakespeare'` and `'Shakespeare'` are different tokens.

In [3]:
def next_token_probs(model, prompt):
    """Probability of each vocabulary token being the next token after the prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    # logits has shape [batch, position, vocabulary]. Keep the last position.
    return torch.softmax(logits[0, -1], dim=-1)


def show_top(model, prompt, k=5):
    probs = next_token_probs(model, prompt)
    top = torch.topk(probs, k)
    for p, token_id in zip(top.values, top.indices):
        print(f"{p.item():.3f}  {tokenizer.decode(token_id)!r}")


show_top(model, "The author of Hamlet is William")

0.511  ' Shakespeare'
0.013  ','
0.009  "'s"
0.009  ' H'
0.009  ' C'


## 4. The scoring rule

**What it does:** turns "does the model know this fact?" into a rule a machine can apply.

- **Correct:** the first token of the right answer is the model's single most likely next token.
- **Confidence:** the probability the model gives to that token.

The experiment needs both. Correct or not decides which facts enter the study. Confidence is what the control uses later to compare facts the healthy model knew equally well.

**What you should see:** `(True, 0.51...)`.

In [4]:
def score_fact(model, prompt, answer):
    """Return (is_correct, confidence) for one fact.

    is_correct: the first token of the answer is the model's top prediction.
    confidence: probability the model gives to that first token.
    """
    probs = next_token_probs(model, prompt)
    first_token_id = tokenizer(answer).input_ids[0]
    is_correct = probs.argmax().item() == first_token_id
    return is_correct, probs[first_token_id].item()


score_fact(model, "The author of Hamlet is William", " Shakespeare")

(True, 0.5108818411827087)

## 5. A tiny exam

**What it does:** applies `score_fact` to a handful of hand-written facts.

**What you should see:** the model gets a couple right and misses most, including "The capital of France is". That is expected. Pythia-160M is a small model and the rule is strict: the answer has to be its number one prediction.

This is why the real exam keeps only the facts the healthy model answers correctly. A fact the model never knew cannot be forgotten.

In [5]:
facts = [
    ("The author of Hamlet is William", " Shakespeare"),
    ("The currency of the United Kingdom is the", " pound"),
    ("The capital of France is", " Paris"),
    ("The capital of Japan is", " Tokyo"),
    ("The Eiffel Tower is located in the city of", " Paris"),
    ("The largest planet in the solar system is", " Jupiter"),
]

for prompt, answer in facts:
    is_correct, confidence = score_fact(model, prompt, answer)
    print(f"{str(is_correct):<5}  {confidence:.3f}  {prompt} ->{answer}")

True   0.511  The author of Hamlet is William -> Shakespeare
True   0.302  The currency of the United Kingdom is the -> pound
False  0.003  The capital of France is -> Paris
False  0.015  The capital of Japan is -> Tokyo
False  0.022  The Eiffel Tower is located in the city of -> Paris
False  0.026  The largest planet in the solar system is -> Jupiter


## 6. Time travel

**What it does:** asks the same fact to four snapshots of the same model: before any training (step 0), early, in the middle, and at the end.

**What you should see:** confidence near zero at step 0, rising as training goes on. The first snapshot from which the model gets a fact right, and keeps getting it right, is what the project calls the "age" of that memory.

In [6]:
prompt, answer = "The author of Hamlet is William", " Shakespeare"

for step in [0, 1000, 16000, 143000]:
    snapshot = load_checkpoint(step)
    is_correct, confidence = score_fact(snapshot, prompt, answer)
    print(f"step {step:>6}  correct={str(is_correct):<5}  confidence={confidence:.3f}")
    del snapshot

model.safetensors: reconstructing file:   0%|          |  0.00B /  649MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

step      0  correct=False  confidence=0.000


model.safetensors: reconstructing file:   0%|          |  0.00B /  649MB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

step  16000  correct=True   confidence=0.042


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

step 143000  correct=True   confidence=0.511


## 7. Where the damage will happen

**What it does:** counts the MLP neurons of the model. In the damage stage, a random percentage of these is switched off.

**What you should see:** 12 layers with 3072 neurons each, 36864 in total.

In [7]:
n_layers = model.config.num_hidden_layers
neurons_per_layer = model.config.intermediate_size
print(f"{n_layers} layers x {neurons_per_layer} neurons = {n_layers * neurons_per_layer} MLP neurons")

12 layers x 3072 neurons = 36864 MLP neurons


## Checkpoint: explain it in your own words

Before moving on, answer these without looking at the code:

1. What is a training snapshot, and how does this notebook pick one?
2. What two things does `score_fact` return, and why does the experiment need both?
3. Why does the study keep only the facts the healthy model answers correctly?